# 02 - Data Preprocessing

This notebook processes raw GeoTIFF exports into pixel-level 3D data cubes.

**Inputs:**
- `data/raw/sentinel1/S1_VV_YYYYMM.tif` - Monthly VV backscatter
- `data/raw/sentinel2/S2_YYYYMM.tif` - Monthly B4 (Red) and B8 (NIR)

**Outputs:**
- `data/processed/inundation_cube.nc` - 3D cube (x, y, time) of binary water masks
- `data/processed/ndvi_cube.nc` - 3D cube (x, y, time) of NDVI values

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import rasterio
from rasterio.enums import Resampling
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Paths
DATA_DIR = Path('data')
RAW_S1_DIR = DATA_DIR / 'raw' / 'sentinel1'
RAW_S2_DIR = DATA_DIR / 'raw' / 'sentinel2'
PROCESSED_DIR = DATA_DIR / 'processed'

# Ensure output directory exists
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

---
## Part 1: Inundation Classification from Sentinel-1

Apply VV threshold to classify water/non-water per pixel per month.

In [ ]:
# Inundation threshold (dB)
# Water typically has VV backscatter < -15 to -18 dB
# Adjust based on local calibration if needed
VV_WATER_THRESHOLD = -16  # dB

In [ ]:
def get_sorted_files(directory, pattern='S1_VV_*.tif'):
    """Get sorted list of files matching pattern."""
    files = sorted(directory.glob(pattern))
    return files

def parse_date_from_filename(filename):
    """Extract date from filename like S1_VV_202006.tif"""
    # Extract YYYYMM from filename
    name = filename.stem  # e.g., 'S1_VV_202006'
    date_str = name.split('_')[-1]  # '202006'
    year = int(date_str[:4])
    month = int(date_str[4:6])
    return datetime(year, month, 1)

# List available Sentinel-1 files
s1_files = get_sorted_files(RAW_S1_DIR, 'S1_VV_*.tif')
print(f"Found {len(s1_files)} Sentinel-1 files")
if s1_files:
    print(f"First: {s1_files[0].name}")
    print(f"Last: {s1_files[-1].name}")

In [ ]:
def load_raster(filepath):
    """Load a GeoTIFF and return data array and metadata."""
    with rasterio.open(filepath) as src:
        data = src.read(1)  # Read first band
        transform = src.transform
        crs = src.crs
        height, width = data.shape
        
        # Get 1D coordinate arrays directly from transform
        # x coords: pixel column centers
        x_coords = np.array([transform[2] + (col + 0.5) * transform[0] for col in range(width)])
        # y coords: pixel row centers
        y_coords = np.array([transform[5] + (row + 0.5) * transform[4] for row in range(height)])
        
    return data, x_coords, y_coords, crs, transform

In [ ]:
def build_inundation_cube(s1_files, threshold):
    """
    Build 3D inundation cube from Sentinel-1 files.
    
    Parameters:
    -----------
    s1_files : list of Path objects
    threshold : float, VV threshold in dB for water classification
    
    Returns:
    --------
    xr.DataArray : 3D cube with dimensions (time, y, x)
    """
    if len(s1_files) == 0:
        raise ValueError("No Sentinel-1 files found")
    
    # Get reference dimensions from first file
    ref_data, x_coords, y_coords, crs, transform = load_raster(s1_files[0])
    n_times = len(s1_files)
    height, width = ref_data.shape
    
    print(f"Building cube: {height} x {width} pixels x {n_times} months")
    
    # Initialize cube
    cube = np.zeros((n_times, height, width), dtype=np.int8)
    times = []
    
    # Process each file
    for i, filepath in enumerate(tqdm(s1_files, desc="Processing S1")):
        # Load VV backscatter
        data, _, _, _, _ = load_raster(filepath)
        
        # Apply threshold: water = 1, non-water = 0
        water_mask = (data < threshold).astype(np.int8)
        
        # Handle nodata (typically very negative or NaN)
        nodata_mask = np.isnan(data) | (data < -50)
        water_mask[nodata_mask] = -1  # Mark as nodata
        
        cube[i, :, :] = water_mask
        times.append(parse_date_from_filename(filepath))
    
    # Create xarray DataArray
    da = xr.DataArray(
        data=cube,
        dims=['time', 'y', 'x'],
        coords={
            'time': pd.DatetimeIndex(times),
            'y': y_coords,
            'x': x_coords
        },
        name='inundation',
        attrs={
            'description': 'Binary inundation mask (1=water, 0=non-water, -1=nodata)',
            'threshold_dB': threshold,
            'source': 'Sentinel-1 GRD VV',
            'crs': str(crs)
        }
    )
    
    return da

In [ ]:
# Build inundation cube
inundation_cube = build_inundation_cube(s1_files, VV_WATER_THRESHOLD)
print(f"\nInundation cube shape: {inundation_cube.shape}")
print(f"Time range: {inundation_cube.time.values[0]} to {inundation_cube.time.values[-1]}")

In [ ]:
# Visualize sample months
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# Select sample months (dry season and wet season)
sample_times = ['2020-02', '2020-06', '2020-10', '2023-02', '2023-06', '2023-10']

for ax, t in zip(axes.flat, sample_times):
    try:
        data = inundation_cube.sel(time=t, method='nearest')
        im = ax.imshow(data.values, cmap='Blues', vmin=0, vmax=1)
        ax.set_title(f'{pd.Timestamp(data.time.values).strftime("%Y-%m")}')
        ax.axis('off')
    except:
        ax.set_title(f'{t} (not available)')
        ax.axis('off')

plt.suptitle('Inundation Maps (Blue = Water)', fontsize=14)
plt.tight_layout()
plt.savefig('outputs/inundation_samples.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save inundation cube
inundation_path = PROCESSED_DIR / 'inundation_cube.nc'
inundation_cube.to_netcdf(inundation_path)
print(f"Saved: {inundation_path}")

### Threshold Sensitivity Analysis

In [ ]:
# Optional: Analyze sensitivity to threshold choice
# Load a sample S1 file to examine VV distribution

if s1_files:
    sample_data, _, _, _, _ = load_raster(s1_files[len(s1_files)//2])  # Middle of time series
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Histogram of VV values
    valid_data = sample_data[sample_data > -50].flatten()
    axes[0].hist(valid_data, bins=100, edgecolor='black', alpha=0.7)
    axes[0].axvline(VV_WATER_THRESHOLD, color='red', linestyle='--', label=f'Threshold: {VV_WATER_THRESHOLD} dB')
    axes[0].set_xlabel('VV Backscatter (dB)')
    axes[0].set_ylabel('Pixel Count')
    axes[0].set_title('VV Backscatter Distribution')
    axes[0].legend()
    
    # Water fraction vs threshold
    thresholds = np.arange(-20, -10, 0.5)
    water_fractions = [(valid_data < t).mean() * 100 for t in thresholds]
    axes[1].plot(thresholds, water_fractions, 'b-o')
    axes[1].axvline(VV_WATER_THRESHOLD, color='red', linestyle='--')
    axes[1].set_xlabel('VV Threshold (dB)')
    axes[1].set_ylabel('Water Fraction (%)')
    axes[1].set_title('Threshold Sensitivity')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('outputs/threshold_sensitivity.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## Part 2: NDVI Calculation from Sentinel-2

Calculate NDVI from B4 (Red) and B8 (NIR) bands.

In [ ]:
# List available Sentinel-2 files
s2_files = get_sorted_files(RAW_S2_DIR, 'S2_*.tif')
print(f"Found {len(s2_files)} Sentinel-2 files")
if s2_files:
    print(f"First: {s2_files[0].name}")
    print(f"Last: {s2_files[-1].name}")

In [ ]:
def load_s2_and_compute_ndvi(filepath):
    """
    Load Sentinel-2 file and compute NDVI.
    
    Expected bands: B4 (Red, band 1), B8 (NIR, band 2)
    """
    with rasterio.open(filepath) as src:
        red = src.read(1).astype(np.float32)  # B4
        nir = src.read(2).astype(np.float32)  # B8
        transform = src.transform
        crs = src.crs
        height, width = red.shape
        
        # Get 1D coordinate arrays directly from transform
        x_coords = np.array([transform[2] + (col + 0.5) * transform[0] for col in range(width)])
        y_coords = np.array([transform[5] + (row + 0.5) * transform[4] for row in range(height)])
    
    # Compute NDVI
    denominator = nir + red
    ndvi = np.where(denominator > 0, (nir - red) / denominator, np.nan)
    ndvi = np.clip(ndvi, -1, 1)
    
    return ndvi, x_coords, y_coords, crs

In [ ]:
def parse_s2_date_from_filename(filename):
    """Extract date from filename like S2_202006.tif"""
    name = filename.stem  # e.g., 'S2_202006'
    date_str = name.split('_')[-1]  # '202006'
    year = int(date_str[:4])
    month = int(date_str[4:6])
    return datetime(year, month, 1)

In [ ]:
def build_ndvi_cube(s2_files):
    """
    Build 3D NDVI cube from Sentinel-2 files.
    
    Returns:
    --------
    xr.DataArray : 3D cube with dimensions (time, y, x)
    """
    if len(s2_files) == 0:
        raise ValueError("No Sentinel-2 files found")
    
    # Get reference dimensions from first file
    ref_ndvi, x_coords, y_coords, crs = load_s2_and_compute_ndvi(s2_files[0])
    n_times = len(s2_files)
    height, width = ref_ndvi.shape
    
    print(f"Building cube: {height} x {width} pixels x {n_times} months")
    
    # Initialize cube
    cube = np.zeros((n_times, height, width), dtype=np.float32)
    times = []
    
    # Process each file
    for i, filepath in enumerate(tqdm(s2_files, desc="Processing S2")):
        ndvi, _, _, _ = load_s2_and_compute_ndvi(filepath)
        cube[i, :, :] = ndvi
        times.append(parse_s2_date_from_filename(filepath))
    
    # Create xarray DataArray
    da = xr.DataArray(
        data=cube,
        dims=['time', 'y', 'x'],
        coords={
            'time': pd.DatetimeIndex(times),
            'y': y_coords,
            'x': x_coords
        },
        name='ndvi',
        attrs={
            'description': 'Normalized Difference Vegetation Index',
            'formula': '(NIR - Red) / (NIR + Red)',
            'source': 'Sentinel-2 SR Harmonized',
            'crs': str(crs)
        }
    )
    
    return da

In [ ]:
# Build NDVI cube
ndvi_cube = build_ndvi_cube(s2_files)
print(f"\nNDVI cube shape: {ndvi_cube.shape}")
print(f"Time range: {ndvi_cube.time.values[0]} to {ndvi_cube.time.values[-1]}")

In [ ]:
# Visualize sample months
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

sample_times = ['2020-02', '2020-06', '2020-10', '2023-02', '2023-06', '2023-10']

for ax, t in zip(axes.flat, sample_times):
    try:
        data = ndvi_cube.sel(time=t, method='nearest')
        im = ax.imshow(data.values, cmap='RdYlGn', vmin=-0.2, vmax=0.8)
        ax.set_title(f'{pd.Timestamp(data.time.values).strftime("%Y-%m")}')
        ax.axis('off')
    except:
        ax.set_title(f'{t} (not available)')
        ax.axis('off')

plt.suptitle('NDVI Maps (Green = High Vegetation)', fontsize=14)
cbar = fig.colorbar(im, ax=axes, orientation='horizontal', fraction=0.05, pad=0.08)
cbar.set_label('NDVI')
plt.savefig('outputs/ndvi_samples.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save NDVI cube
ndvi_path = PROCESSED_DIR / 'ndvi_cube.nc'
ndvi_cube.to_netcdf(ndvi_path)
print(f"Saved: {ndvi_path}")

---
## Summary Statistics

In [ ]:
# Summary of processed data
print("=" * 50)
print("DATA PREPROCESSING SUMMARY")
print("=" * 50)
print(f"\nInundation Cube:")
print(f"  Shape: {inundation_cube.shape}")
print(f"  Time range: {pd.Timestamp(inundation_cube.time.values[0]).strftime('%Y-%m')} to {pd.Timestamp(inundation_cube.time.values[-1]).strftime('%Y-%m')}")
print(f"  Threshold: {VV_WATER_THRESHOLD} dB")
print(f"  File: {inundation_path}")

print(f"\nNDVI Cube:")
print(f"  Shape: {ndvi_cube.shape}")
print(f"  Time range: {pd.Timestamp(ndvi_cube.time.values[0]).strftime('%Y-%m')} to {pd.Timestamp(ndvi_cube.time.values[-1]).strftime('%Y-%m')}")
print(f"  Value range: {np.nanmin(ndvi_cube.values):.2f} to {np.nanmax(ndvi_cube.values):.2f}")
print(f"  File: {ndvi_path}")

In [ ]:
# Time series of aggregated metrics (preview for breakpoint detection)
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# Monthly inundation extent
valid_mask = inundation_cube >= 0  # Exclude nodata
monthly_inundation = (inundation_cube.where(valid_mask) == 1).sum(dim=['x', 'y']) / valid_mask.sum(dim=['x', 'y']) * 100
axes[0].plot(inundation_cube.time, monthly_inundation, 'b-', linewidth=1)
axes[0].set_ylabel('Inundation Extent (%)')
axes[0].set_title('Monthly Inundation Extent')
axes[0].grid(True, alpha=0.3)

# Monthly mean NDVI
monthly_ndvi = ndvi_cube.mean(dim=['x', 'y'])
axes[1].plot(ndvi_cube.time, monthly_ndvi, 'g-', linewidth=1)
axes[1].set_ylabel('Mean NDVI')
axes[1].set_xlabel('Date')
axes[1].set_title('Monthly Mean NDVI')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/time_series_preview.png', dpi=150, bbox_inches='tight')
plt.show()

## Next Steps

1. Proceed to `03_breakpoint_detection.ipynb` to identify the regime transition timing
2. The breakpoint will define pre/post periods for subsequent analyses